## Running Athena++ on AWS Parallel Computing Service (PCS)

This notebook runs the **same Orszag-Tang MHD simulation** as the AWS ParallelCluster lab, but on **AWS Parallel Computing Service (AWS PCS)** - a *managed* Slurm service.

The contrast with ParallelCluster is the point of the lab:

| | ParallelCluster | PCS (this notebook) |
|---|---|---|
| Slurm controller | A head node **you** run | **Managed by AWS** |
| Define compute | Config file | **Compute node groups** attached to the cluster |
| Partition | Config file | **Queue** attached to the cluster |
| Create / manage | `pcluster` CLI | PCS API (boto3) |
| Job submission | `sbatch` | **identical** `sbatch` |
| Visualization | VTK from S3 | **identical** |

We will:
1. Create a PCS cluster (managed Slurm controller).
2. Attach a **login** node group and a **4-node compute** node group, plus a **queue** named `big`.
3. Build Athena++ (VTK output) onto a shared EFS filesystem mounted at `/shared`.
4. Submit the Orszag-Tang job with `sbatch` and monitor it.
5. Download the VTK output from S3 and visualize it - the same viz as the ParallelCluster lab.


## Load the PCS helper module

In [ ]:
import boto3
import botocore
import json
import os
import sys
import time
import importlib
import project_path  # adds repo root to sys.path (for lib.workshop)
from botocore.exceptions import ClientError

try:
    del sys.modules['pcs_helper']
except KeyError:
    pass
import pcs_helper
importlib.reload(pcs_helper)
from pcs_helper import PCSHelper

my_session = boto3.session.Session()
REGION = my_session.region_name
print("Region:", REGION)

# ---- settings ----
cluster_name = 'pcs-athena'
slurm_version = '25.11'                 # must match an available PCS sample AMI
compute_instance_type = 'c5n.2xlarge'   # same as the ParallelCluster 'big' queue
login_instance_type = 'c5.xlarge'
compute_max_nodes = 4                   # 4-node compute node group

# build/ holds the rendered input + batch files
os.makedirs('build', exist_ok=True)

helper = PCSHelper(cluster_name, slurm_version=slurm_version,
                   compute_instance_type=compute_instance_type,
                   login_instance_type=login_instance_type,
                   compute_max_nodes=compute_max_nodes)


## Step 1 - Create the prerequisites

This creates (idempotently): an S3 bucket for job files/output, a security group, an **EFS** filesystem mounted at `/shared`, a PCS node **IAM instance profile** (path `/aws-pcs/` with `pcs:RegisterComputeNodeGroupInstance`), and two **EC2 launch templates** (login + compute) whose user-data mounts EFS. The **login** launch template also builds Athena++ into `/shared`.

It also resolves the newest PCS **sample AMI** for Slurm `25.11` via `describe-images`.

In [ ]:
helper.create_before()

## Step 2 - Create the PCS cluster

The cluster is the managed Slurm controller. Creation takes several minutes; the helper waits until it is `ACTIVE`. 

This step takes around 10 minutes.

In [ ]:
%%time
helper.create_cluster()

## Step 3 - Create the node groups and queue

- **login** node group (1 node): interactive access + where Athena++ is built into `/shared`.
- **compute** node group (0 -> up to 4 nodes): where jobs run; mirrors the ParallelCluster `big` queue (`c5n.2xlarge`).
- **queue `big`**: the Slurm partition that routes `sbatch -p big` jobs to the compute node group.

This step takes around 5 minutes. 

In [ ]:
%%time
helper.create_node_groups()
helper.create_queue('big')

## Step 4 - Prepare the simulation input and batch script

Same templating as the ParallelCluster lab. The mesh is decomposed into meshblocks; the number of meshblocks determines how many MPI ranks (cores) the job uses. With `c5n.2xlarge` (4 cores/node) and a 256x256 mesh in 64x64 meshblocks we get 4x4 = 16 meshblocks -> 16 ranks -> 4 nodes x 4 cores.

The batch script writes **VTK** output and copies results to S3.

In [ ]:
# ----- simulation settings -----
simulation_name = 'orz-256x256'
num_cores_per_node = 4          # c5n.2xlarge physical cores
partition = 'big'               # the PCS queue we created
account_name = 'dept-2d'
use_efa = 'NO'
job_name = f'{simulation_name}-{compute_instance_type}'
my_prefix = 'athena/' + job_name

# template + output file names
input_file_ini = 'config/athinput_orszag_tang.ini'
batch_file_ini = 'config/batch_athena_sh.ini'
input_file = 'athinput_orszag_tang.input'
batch_file = 'batch_athena.sh'

# ----- mesh / meshblock -----
nx1, nx2, nx3 = 256, 256, 1
mbnx1, mbnx2, mbnx3 = 64, 64, 1
num_of_threads = 1
num_blocks = (nx1/mbnx1) * (nx2/mbnx2) * (nx3/mbnx3)   # 16
num_nodes = int(num_blocks / num_cores_per_node)        # 4
num_tasks_per_node = int(num_blocks / num_nodes / num_of_threads)

exe_path = '/shared/athena-public-version/bin/athena'
work_dir = '/shared/tmp/' + job_name

ph = {'${nx1}': str(nx1), '${nx2}': str(nx2), '${nx3}': str(nx3),
      '${mbnx1}': str(mbnx1), '${mbnx2}': str(mbnx2), '${mbnx3}': str(mbnx3),
      '${num_of_threads}': str(num_of_threads)}
helper.template_to_file(input_file_ini, 'build/' + input_file, ph)

ph = {'${nodes}': str(num_nodes),
      '${ntasks-per-node}': str(num_tasks_per_node),
      '${cpus-per-task}': str(num_of_threads),
      '${account}': account_name,
      '${partition}': partition,
      '${job-name}': job_name,
      '${EXE_PATH}': exe_path,
      '${WORK_DIR}': work_dir,
      '${input-file}': input_file,
      '${BUCKET_NAME}': helper.my_bucket_name,
      '${PREFIX}': my_prefix,
      '${USE_EFA}': use_efa,
      '${OUTPUT_FOLDER}': 'output/',
      '${NUM_OF_THREADS}': str(num_of_threads)}
helper.template_to_file(batch_file_ini, 'build/' + batch_file, ph)

helper.upload_athena_files(input_file, batch_file, my_prefix)

# Also upload the Athena++ build script so it can be run on the login node if the
# launch-time build didn't finish (see Step 5).
boto3.client('s3').upload_file('scripts/install_athena_pcs.sh', helper.my_bucket_name,
                               my_prefix + '/install_athena_pcs.sh')
print('Mesh blocks:', int(num_blocks), '-> nodes:', num_nodes, 'tasks/node:', num_tasks_per_node)
print('Bucket:', helper.my_bucket_name, 'prefix:', my_prefix)


## Step 5 - Connect to the login node and submit the job

PCS has no head node you SSH into; instead connect to the **login** node group instance with **Systems Manager Session Manager**:

1. In the console, open **EC2 -> Instances** and find the running instance for the `login` node group (or **Systems Manager -> Session Manager -> Start session**).
2. Start a session, then run:

```bash
sudo su - ec2-user

# 1. Confirm Athena++ built into the shared filesystem (should be a non-empty executable)
ls -l /shared/athena-public-version/bin/athena

# If it is MISSING or ZERO BYTES, the launch-time build did not finish. Check the log
#   sudo cat /var/log/pcs-bootstrap.log
# and (re)build it directly on the login node:
#   aws s3 cp s3://BUCKET/PREFIX/install_athena_pcs.sh .   # or copy from the repo
#   chmod +x install_athena_pcs.sh && ./install_athena_pcs.sh
# The script locates mpicxx under /opt/amazon/openmpi (the PCS AMI ships EFA OpenMPI
# there; 'module load openmpi' is not configured, which is why a plain 'make' fails
# with 'mpicxx: No such file or directory').

# 2. Fetch the batch script + input from S3 and submit to the 'big' queue
sudo mkdir -p /shared/tmp/JOB && sudo chown -R /shared/tmp ec2-user:ec2user && cd /shared/tmp/JOB
aws s3 cp s3://BUCKET/PREFIX/batch_athena.sh .
aws s3 cp s3://BUCKET/PREFIX/athinput_orszag_tang.input .
sbatch -p big batch_athena.sh
```

Replace `BUCKET`/`PREFIX`/`JOB` with the values printed above. Monitor with the **same Slurm commands** as ParallelCluster:

```bash
squeue      # job queue
sinfo       # partitions / node states (compute scales 0 -> 4 on demand)
```

The compute node group scales up from 0 when the job is submitted, so the first job waits a few minutes for nodes to launch. The batch script writes VTK output and uploads it to `s3://BUCKET/PREFIX/output/`.


### A note on the login-node build

The Athena++ build runs once on the login node's first boot (see `/var/log/pcs-bootstrap.log` on the login instance). If `ls /shared/athena-public-version/bin/athena` is missing, the build is still running or failed - check that log.

# Visualize the Athena++ simulation results

In [ ]:
%pip install -q matplotlib

In [ ]:
data_folder = job_name + '/output'
output_folder = helper.my_bucket_name + '/' + my_prefix + '/output'
if not os.path.isdir(job_name):
    !mkdir -p $job_name
else:
    !rm -rf $job_name/*
!aws s3 cp s3://$output_folder/ ./$data_folder/ --recursive


In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
# Athena++ ships a VTK reader (athena_read) under vis/python
if not os.path.isdir('athena-public-version'):
    !git clone https://github.com/PrincetonUniversity/athena-public-version
sys.path.insert(0, 'athena-public-version/vis/python')
import athena_read


### Read the history (hst) data

History data shows overall parameter changes over time.

In [ ]:
%matplotlib inline
hst = athena_read.hst(data_folder + '/OrszagTang.hst')
num_timesteps = len(hst['time'])
print(hst.keys())
plt.plot(hst['time'], hst['dt'])


### Read the VTK output and animate

`athena_read.vtk()` returns a 4-tuple `(x_faces, y_faces, z_faces, data)` where `data` is a dict keyed by field name (`rho`, `press`, `vel1`, `Bcc1`, ...). Each field is a 3D array indexed `[k, j, i]` (z, y, x).

In [ ]:
def process_vtk(filename):
    _, _, _, data = athena_read.vtk(filename)
    return data

def read_all_timestep(template, num_steps, field_names, slice_number, dimension):
    assert dimension in ('x', 'y', 'z')
    result = {f: [] for f in field_names}
    for i in range(num_steps):
        fn = template.format(str(i).zfill(5))
        data = process_vtk(fn)
        for f in field_names:
            if dimension == 'x':
                result[f].append(data[f][slice_number, :, :])
            elif dimension == 'y':
                result[f].append(data[f][:, slice_number, :])
            else:
                result[f].append(data[f][:, :, slice_number])
    return result

def animate_slice(frames):
    plt.figure()
    for i in range(len(frames)):
        plt.imshow(frames[i])
        plt.title('Frame %d' % i)
        plt.show()
        plt.pause(0.2)
        clear_output(wait=True)


In [ ]:
data_file_name_template = data_folder + '/OrszagTang.joined.out2.{}.vtk'
data = read_all_timestep(data_file_name_template, num_timesteps, ['press', 'rho'], 0, 'x')


In [ ]:
# Pressure over time
animate_slice(data['press'])

In [ ]:
# Density over time
animate_slice(data['rho'])

# Clean up

Delete the PCS resources (queue -> node groups -> cluster) when done. Set `delete_efs=True` to also remove the shared filesystem.

In [ ]:
helper.cleanup(delete_efs=False)